# A1 – BloodMNIST exploration

In [ ]:
import hashlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start):
    for folder in (start, *start.parents):
        if (folder / "src" / "data.py").is_file():
            return folder
    raise FileNotFoundError(f"No folder containing src/data.py found at or above {start}.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "src"))

from data import CLASS_NAMES, CLASS_SHORT_NAMES, RESOLUTIONS, load_all

SEED = 0
SPLITS = ("train", "val", "test")
SPLIT_PAIRS = (("train", "val"), ("train", "test"), ("val", "test"))
FIGURES_DIR = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

rng = np.random.default_rng(SEED)
pd.set_option("display.precision", 4)

## 1. Load

In [ ]:
data = {res: load_all(res, deduplicate=False) for res in RESOLUTIONS}

## 2. Shapes and values

In [ ]:
rows = []
for res in RESOLUTIONS:
    for split in SPLITS:
        images, _, _ = data[res][split]
        rows.append({
            "resolution": res,
            "split": split,
            "n_images": len(images),
            "image_shape": images.shape[1:],
            "dtype": str(images.dtype),
            "min": int(images.min()),
            "max": int(images.max()),
        })
pd.DataFrame(rows)

In [ ]:
for split in SPLITS:
    reference = data[RESOLUTIONS[0]][split][1]
    same = all(np.array_equal(reference, data[res][split][1]) for res in RESOLUTIONS[1:])
    print(f"{split}: labels identical across resolutions: {same}")

## 3. Class distribution

Labels are identical across resolutions, so counts use 28 px.

In [ ]:
counts = pd.DataFrame(
    {split: np.bincount(data[28][split][1], minlength=len(CLASS_NAMES)) for split in SPLITS},
    index=pd.Index(CLASS_SHORT_NAMES, name="class"),
)
percentages = counts / counts.sum() * 100
distribution = pd.concat({"count": counts, "percent": percentages.round(2)}, axis=1)
distribution

In [ ]:
imbalance_ratio = counts["train"].max() / counts["train"].min()
print(f"Imbalance ratio (train, largest / smallest): {imbalance_ratio:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(counts.index, counts["train"], color="tab:blue")
ax.set_ylabel("Number of images")
ax.set_title("BloodMNIST train split: images per class")
ax.tick_params(axis="x", rotation=30)
plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "A1_class_distribution.png", dpi=300)
plt.show()

## 4. Sample grid

The same train image per class, shown at each resolution.

In [ ]:
train_labels = data[28]["train"][1]
sample_indices = [rng.choice(np.flatnonzero(train_labels == c)) for c in range(len(CLASS_NAMES))]

fig, axes = plt.subplots(len(CLASS_NAMES), len(RESOLUTIONS), figsize=(6, 12))
for row, (index, name) in enumerate(zip(sample_indices, CLASS_SHORT_NAMES)):
    for col, res in enumerate(RESOLUTIONS):
        ax = axes[row, col]
        ax.imshow(data[res]["train"][0][index], interpolation="nearest")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f"{res} px")
        if col == 0:
            ax.set_ylabel(name, rotation=0, ha="right", va="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "A1_sample_grid.png", dpi=300)
plt.show()

## 5. Image statistics

Per-channel mean and standard deviation per class, train split at 128 px, pixel values scaled to 0–1.

In [ ]:
images, labels, _ = data[128]["train"]
rows = []
for c, name in enumerate(CLASS_SHORT_NAMES):
    class_images = images[labels == c]
    row = {"class": name, "n_images": len(class_images)}
    # One channel at a time keeps the float copy small.
    for ch, channel in enumerate("RGB"):
        values = class_images[..., ch].astype(np.float32) / 255
        row[f"mean_{channel}"] = values.mean(dtype=np.float64)
        row[f"std_{channel}"] = values.std(dtype=np.float64)
    rows.append(row)
channel_stats = pd.DataFrame(rows).set_index("class")
channel_stats

## 6. Data checks

Near-blank: per-image pixel standard deviation below 2 (0–255 scale). Duplicates: identical MD5 hashes of the image bytes.

In [ ]:
BLANK_STD_THRESHOLD = 2


def image_hashes(images):
    return [hashlib.md5(image.tobytes()).hexdigest() for image in images]


near_blank_rows = []
within_rows = []
across_rows = []
for res in RESOLUTIONS:
    hashes = {}
    for split in SPLITS:
        images = data[res][split][0]
        pixel_std = np.array([image.std() for image in images])
        near_blank_rows.append({
            "resolution": res,
            "split": split,
            "near_blank": int((pixel_std < BLANK_STD_THRESHOLD).sum()),
        })
        hashes[split] = image_hashes(images)
        within_rows.append({
            "resolution": res,
            "split": split,
            "duplicates": len(images) - len(set(hashes[split])),
        })
    for a, b in SPLIT_PAIRS:
        shared = set(hashes[a]) & set(hashes[b])
        across_rows.append({
            "resolution": res,
            "pair": f"{a}-{b}",
            "shared_images": len(shared),
        })

near_blank = pd.DataFrame(near_blank_rows).pivot(index="resolution", columns="split", values="near_blank")[list(SPLITS)]
within_duplicates = pd.DataFrame(within_rows).pivot(index="resolution", columns="split", values="duplicates")[list(SPLITS)]
across_duplicates = pd.DataFrame(across_rows).pivot(index="resolution", columns="pair", values="shared_images")[[f"{a}-{b}" for a, b in SPLIT_PAIRS]]
print("Near-blank images:")
display(near_blank)
print("Duplicates within split (images minus unique images):")
display(within_duplicates)
print("Unique images shared between splits:")
display(across_duplicates)

## 7. Summary

In [ ]:
split_sizes = counts.sum()
print("Images per split:", ", ".join(f"{split} {n}" for split, n in split_sizes.items()))
print("Total images:", split_sizes.sum())
print(f"Imbalance ratio (train): {imbalance_ratio:.2f}")
for res in RESOLUTIONS:
    print(
        f"{res} px | near-blank: {near_blank.loc[res].to_dict()}"
        f" | duplicates within split: {within_duplicates.loc[res].to_dict()}"
        f" | shared across splits: {across_duplicates.loc[res].to_dict()}"
    )